# 🏗️ Pipeline de Preprocesamiento (Sprint 2)

## 🎯 Objetivo

Construir un pipeline reproducible que integre:

- Data Cleaning (Notebook 05)
- Feature Engineering (Notebook 06)
- Preparación para modelado (Notebook 07)

## ⚙️ Tecnologías

- scikit-learn Pipeline
- ColumnTransformer
- joblib

---

## 🚨 Regla clave

El pipeline debe:

✔ Entrenarse con `.fit()`  
✔ Aplicarse con `.transform()`  
✔ Evitar data leakage  
✔ Ser reutilizable en modelado (Sprint 3 y 4)

In [2]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

import joblib

In [10]:
import os

ruta = '../data/raw/'
print(os.listdir(ruta))

['01-hotel_bookings.csv.dvc']


## 📂 Carga de datos crudos

In [15]:
# ============================================================
# 📂 CARGA DE DATOS CRUDOS
# ============================================================

import pandas as pd

df = pd.read_csv('../data/01-hotel_bookings.csv')

print("Shape original:", df.shape)
df.head()

Shape original: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,03-07-15


## 🧹 Paso 1: Data Cleaning

Aplicamos las reglas definidas en el Notebook 05:
- Eliminación de leakage
- Eliminación de duplicados
- Corrección de nulos
- Eliminación de registros inválidos

In [16]:
# ============================================================
# DATA CLEANING
# ============================================================

df_clean = df.copy()

# eliminar leakage
df_clean.drop(['reservation_status', 'reservation_status_date', 'company'], axis=1, inplace=True)

# duplicados
df_clean = df_clean.drop_duplicates()

# sin personas
df_clean = df_clean[~(
    (df_clean['adults'] == 0) &
    (df_clean['children'].fillna(0) == 0) &
    (df_clean['babies'] == 0)
)]

# sin noches
df_clean = df_clean[~(
    (df_clean['stays_in_week_nights'] == 0) &
    (df_clean['stays_in_weekend_nights'] == 0)
)]

# nulos
df_clean['children'] = df_clean['children'].fillna(0)
df_clean['country'] = df_clean['country'].fillna(df_clean['country'].mode()[0])
df_clean['agent'] = df_clean['agent'].fillna(0)

# adr
df_clean = df_clean[df_clean['adr'] >= 0]
df_clean['adr'] = df_clean['adr'].clip(upper=df_clean['adr'].quantile(0.99))

print("Shape después de cleaning:", df_clean.shape)

Shape después de cleaning: (86372, 29)


## 🧠 Paso 2: Feature Engineering

Se generan variables derivadas para mejorar la capacidad predictiva del modelo.

In [17]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

df_feat = df_clean.copy()

df_feat['total_nights'] = df_feat['stays_in_week_nights'] + df_feat['stays_in_weekend_nights']
df_feat['total_guests'] = df_feat['adults'] + df_feat['children'] + df_feat['babies']
df_feat['total_price'] = df_feat['adr'] * df_feat['total_nights']
df_feat['price_per_person'] = df_feat['adr'] / df_feat['total_guests'].replace(0, np.nan)

df_feat['is_loyal'] = (df_feat['is_repeated_guest'] == 1).astype(int)

df_feat['adr_log'] = np.log1p(df_feat['adr'])
df_feat['lead_time_log'] = np.log1p(df_feat['lead_time'])

print("Shape después de FE:", df_feat.shape)

Shape después de FE: (86372, 36)


## 🎯 Separación de variables

In [18]:
# ============================================================
# X / y
# ============================================================

X = df_feat.drop('is_canceled', axis=1)
y = df_feat['is_canceled']

## ⚙️ Paso 3: Construcción del Pipeline

Se construye un pipeline usando:

- ColumnTransformer
- Imputación
- Escalado
- Encoding

In [19]:
# ============================================================
# DEFINIR COLUMNAS
# ============================================================

num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include='object').columns

C:\Users\Renzo\AppData\Local\Temp\ipykernel_24452\1290103767.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


In [20]:
# ============================================================
# PIPELINE NUMÉRICO
# ============================================================

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [21]:
# ============================================================
# PIPELINE CATEGÓRICO
# ============================================================

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [22]:
# ============================================================
# COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

## 🚀 Validación del pipeline

En esta sección se valida que el pipeline funciona correctamente con `.fit()` y `.transform()`.

⚠️ Importante:  
Este entrenamiento es solo para verificación técnica.  
El entrenamiento real se realizará en el flujo de modelado (Notebook 09) usando únicamente el conjunto de entrenamiento, evitando data leakage.

In [23]:
preprocessor.fit(X)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## Validación de transformación del pipeline

Se aplica `.transform()` sobre el dataset completo únicamente para verificar que el pipeline funciona correctamente.

⚠️ Importante:  
Esta transformación es solo para validación técnica.  
En el flujo de modelado (Notebook 09), el pipeline será entrenado con el conjunto de entrenamiento y aplicado posteriormente sobre train y test, evitando data leakage.

In [24]:
# ============================================================
# VALIDACIÓN DE TRANSFORMACIÓN
# ============================================================

X_transformed = preprocessor.transform(X)

print("Transformación aplicada correctamente")
print("Shape transformado:", X_transformed.shape)

Transformación aplicada correctamente
Shape transformado: (86372, 261)


## 💾 Guardado del pipeline

In [25]:
# ============================================================
# GUARDAR PIPELINE (VERSIÓN ROBUSTA)
# ============================================================

import os
import joblib

# Crear carpeta si no existe
os.makedirs('../models', exist_ok=True)

# Guardar pipeline
joblib.dump(preprocessor, '../models/preprocessor.pkl')

print("✅ Pipeline guardado en models/")

✅ Pipeline guardado en models/


## ✅ Conclusión

Se construyó un pipeline completo que:

✔ Integra cleaning + feature engineering  
✔ Es reproducible  
✔ Evita data leakage  
✔ Está listo para ser usado en modelos (Sprint 3)

👉 Este pipeline será utilizado en:

- 09_baseline_models.ipynb
- 10_evaluation.ipynb